# EDA Analysis

Feb 12, 2026

Purpose: explore data and create some static figures

In [1]:
import altair as alt
import pandas as pd

In [2]:
country = pd.read_csv("../data/raw/GlobalLandTemperaturesByCountry.csv")

country.info()

<class 'pandas.DataFrame'>
RangeIndex: 577462 entries, 0 to 577461
Data columns (total 4 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   dt                             577462 non-null  str    
 1   AverageTemperature             544811 non-null  float64
 2   AverageTemperatureUncertainty  545550 non-null  float64
 3   Country                        577462 non-null  str    
dtypes: float64(2), str(2)
memory usage: 17.6 MB


In [3]:
globe = pd.read_csv('../data/raw/GlobalTemperatures.csv')

globe.info()

<class 'pandas.DataFrame'>
RangeIndex: 3192 entries, 0 to 3191
Data columns (total 9 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   dt                                         3192 non-null   str    
 1   LandAverageTemperature                     3180 non-null   float64
 2   LandAverageTemperatureUncertainty          3180 non-null   float64
 3   LandMaxTemperature                         1992 non-null   float64
 4   LandMaxTemperatureUncertainty              1992 non-null   float64
 5   LandMinTemperature                         1992 non-null   float64
 6   LandMinTemperatureUncertainty              1992 non-null   float64
 7   LandAndOceanAverageTemperature             1992 non-null   float64
 8   LandAndOceanAverageTemperatureUncertainty  1992 non-null   float64
dtypes: float64(8), str(1)
memory usage: 224.6 KB


In [4]:
globe["dt"] = pd.to_datetime(globe["dt"])

globe['year'] = globe['dt'].dt.year

year_avg = (
    globe.groupby(['year'], as_index=False)
    .agg(avg_temp = ('LandAverageTemperature', 'mean'),
         avg_uncertainty = ('LandAverageTemperatureUncertainty', 'mean'))
)

year_avg["temp_lower"] = year_avg["avg_temp"] - year_avg["avg_uncertainty"]

year_avg["temp_upper"] = year_avg["avg_temp"] + year_avg["avg_uncertainty"]

year_avg.head()

,year,avg_temp,avg_uncertainty,temp_lower,temp_upper
0,1750,8.719364,2.637818,6.081545,11.357182
1,1751,7.976143,2.781143,5.195000,10.757286
2,1752,5.779833,2.977000,2.802833,8.756833
3,1753,8.388083,3.176000,5.212083,11.564083
4,1754,8.469333,3.494250,4.975083,11.963583


In [5]:
year_avg.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   year             266 non-null    int32  
 1   avg_temp         266 non-null    float64
 2   avg_uncertainty  266 non-null    float64
 3   temp_lower       266 non-null    float64
 4   temp_upper       266 non-null    float64
dtypes: float64(4), int32(1)
memory usage: 9.5 KB


In [6]:
line = alt.Chart(year_avg).mark_line().encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format='.0f')),
    y=alt.Y("avg_temp:Q", title="Temperature (°C)"),
)

band = alt.Chart(year_avg).mark_area(opacity=0.25).encode(
    x=alt.X("year:Q", axis=alt.Axis(format='.0f')),
    y=alt.Y('temp_lower:Q'),
    y2=('temp_upper:Q')
)

chart = (band + line).properties(
    width=900,
    height=450,
    title='Global Land Average Temperature by Year (with Avg Uncertainty)'
)

chart

alt.LayerChart(...)

In [7]:
chart.save('../data/figures/global_land_avg_temp_by_year.png')

In [8]:
country["dt"] = pd.to_datetime(country["dt"])

country["year"] = country["dt"].dt.year

year_country = country.groupby(["year", "Country"], as_index=False).agg(
    avg_temp=("AverageTemperature", "mean"),
    avg_uncertainty=("AverageTemperatureUncertainty", "mean"),
)

year_country["temp_lower"] = year_country["avg_temp"] - year_country["avg_uncertainty"]

year_country["temp_upper"] = year_country["avg_temp"] + year_country["avg_uncertainty"]

year_country.head()

,year,Country,avg_temp,avg_uncertainty,temp_lower,temp_upper
0,1743,Albania,8.620,2.268,6.352,10.888
1,1743,Andorra,7.556,2.188,5.368,9.744
2,1743,Austria,2.482,2.116,0.366,4.598
3,1743,Belarus,0.767,2.465,-1.698,3.232
4,1743,Belgium,7.106,1.855,5.251,8.961


In [9]:
year_country['Country'].unique()

<StringArray>
[                            'Albania',                             'Andorra',
                             'Austria',                             'Belarus',
                             'Belgium',              'Bosnia And Herzegovina',
                            'Bulgaria',                             'Croatia',
                      'Czech Republic',                    'Denmark (Europe)',
 ...
                    'Papua New Guinea',                        'Kingman Reef',
                            'Kiribati',                       'Palmyra Atoll',
      'Federated States Of Micronesia',                                'Guam',
            'Northern Mariana Islands', 'French Southern And Antarctic Lands',
   'Heard Island And Mcdonald Islands',                          'Antarctica']
Length: 243, dtype: str

In [10]:
select_country = "United Kingdom"

input_country = year_country.copy()

input_country = input_country[input_country["Country"] == select_country]

line = (
    alt.Chart(input_country)
    .mark_line()
    .encode(
        x=alt.X("year:Q", title="Year", axis=alt.Axis(format=".0f")),
        y=alt.Y("avg_temp:Q", title="Temperature (°C)"),
    )
)

band = (
    alt.Chart(input_country)
    .mark_area(opacity=0.25)
    .encode(
        x=alt.X("year:Q", axis=alt.Axis(format=".0f")),
        y=alt.Y("temp_lower:Q"),
        y2=("temp_upper:Q"),
    )
)

chart = (band + line).properties(
    width=900,
    height=450,
    title=f"Average Temperature by Year (with Avg Uncertainty) in {select_country}",
)

chart

alt.LayerChart(...)

In [11]:
chart.save("../data/figures/UK_avg_temp_by_year.png")

In [12]:
select_country = "Canada"

input_country = year_country.copy()

input_country = input_country[input_country["Country"] == select_country]

line = (
    alt.Chart(input_country)
    .mark_line()
    .encode(
        x=alt.X("year:Q", title="Year", axis=alt.Axis(format=".0f")),
        y=alt.Y("avg_temp:Q", title="Temperature (°C)"),
    )
)

band = (
    alt.Chart(input_country)
    .mark_area(opacity=0.25)
    .encode(
        x=alt.X("year:Q", axis=alt.Axis(format=".0f")),
        y=alt.Y("temp_lower:Q"),
        y2=("temp_upper:Q"),
    )
)

chart = (band + line).properties(
    width=900,
    height=450,
    title=f"Average Temperature by Year (with Avg Uncertainty) in {select_country}",
)

chart

alt.LayerChart(...)

In [13]:
chart.save("../data/figures/avg_temp_by_year_Canada.png")

In [14]:
overview = pd.DataFrame(
    [
        {
            "Dataset": "country",
            "Rows": len(country),
            "Num Columns": country.shape[1],
            "Scale": "Monthly",
            "Unique Countries": country["Country"].nunique(),
            "Date Range": f"{country['dt'].min().year} to {country['dt'].max().year}",
            "Available Years of Data": globe["dt"].dt.year.nunique(),
            "Missing Temp": country["AverageTemperature"].isna().sum(),
        },
        {
            "Dataset": "globe",
            "Rows": len(globe),
            "Num Columns": globe.shape[1],
            "Scale": "Monthly",
            "Unique Countries": "Global Measurement",
            "Date Range": f"{globe['dt'].min().year} to {globe['dt'].max().year}",
            "Available Years of Data": globe["dt"].dt.year.nunique(),
            "Missing Temp": globe["LandAverageTemperature"].isna().sum(),
        },
    ]
)

# Save + print markdown
overview_md = overview.to_markdown(index=False)
overview_md

'| Dataset   |   Rows |   Num Columns | Scale   | Unique Countries   | Date Range   |   Available Years of Data |   Missing Temp |\n|:----------|-------:|--------------:|:--------|:-------------------|:-------------|--------------------------:|---------------:|\n| country   | 577462 |             5 | Monthly | 243                | 1743 to 2013 |                       266 |          32651 |\n| globe     |   3192 |            10 | Monthly | Global Measurement | 1750 to 2015 |                       266 |             12 |'

In [15]:
overview

,Dataset,Rows,Num Columns,Scale,Unique Countries,Date Range,Available Years of Data,Missing Temp
0,country,577462,5,Monthly,243,1743 to 2013,266,32651
1,globe,3192,10,Monthly,Global Measurement,1750 to 2015,266,12
